[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://githubtocolab.com/Soliman2020/python_48h_notebooks/blob/master/Session_10_Classes_and_Decorators.ipynb)

# Session 10 — Classes, Data Classes, and Decorators
## Python Fundamentals & Practical Applications — 48-Hour Edition

**STUDENT EDITION — WRITE YOUR OWN CODE**  
**Duration:** 4 classroom hours, including a 30-minute break  
**Audience:** beginners who have completed the earlier Python sessions

> **Beginner promise:** Model familiar work items as objects, describe data clearly with data classes, and add reusable behavior with decorators.
> **Professional habit:** Keep business rules in one place and make reusable behavior explicit.

In [ ]:
student_name = input("Enter your name: ")
print(f"{student_name}, welcome to the classes, data classes, and decorators session.")

## Learning outcomes

By the end of this session, you should be able to:

1. Define a class with `__init__` and create objects.
2. Use instance attributes, class attributes, methods, and `__str__`.
3. Keep validation beside the method that changes state.
4. Write and apply a decorator using `@wraps`, `*args`, and `**kwargs`.
5. Explain how multiple decorators are applied.
6. Use `@dataclass`.
7. Combine classes and decorators in a small work-item system.

**Total: 240 minutes.** The retry and cache extensions are challenge work.

## 1. Classes and objects

A **class** is a blueprint. An **object** is one concrete instance created from that blueprint. A class groups related data and behavior.

- An **attribute** stores data: `item.status`.
- A **method** performs an action: `item.close()`.
- `self` refers to the current object.
- `__init__` initializes a new object.
- `__str__` controls how an object is displayed by `print()`.

In [ ]:
class WorkItem:
    """One work item with a small set of business rules."""

    open_statuses = {"open", "in progress"}

    def __init__(self, case_id, sector, hours_open=0, status="open"):
        self.case_id = case_id
        self.sector = sector
        self.hours_open = hours_open
        self.status = status

    def close(self):
        self.status = "closed"
        return self.status

    def is_overdue(self, sla_hours):
        return self.hours_open > sla_hours and self.status in self.open_statuses

    def __str__(self):
        return f"{self.case_id} | {self.sector} | {self.status} | {self.hours_open} hours"

In [ ]:
item = WorkItem("A101", "Finance", hours_open=30)
print(item)
print("Case ID:", item.case_id)
print("Overdue at 24 hours:", item.is_overdue(24))

item.close()
print("After closing:", item)
assert item.status == "closed"
assert not item.is_overdue(24)

### A second example: a class with validated methods

Validation belongs inside the method that changes state. This prevents every caller from having to remember the same rule.

In [ ]:
class BankAccount:
    def __init__(self, account_number, holder, balance=0.0, currency="EUR"):
        self.account_number = account_number
        self.holder = holder
        self.balance = balance
        self.currency = currency

    def deposit(self, amount):
        if amount < 0:
            raise ValueError("Deposit amount must be non-negative.")
        self.balance += amount
        return self.balance

    def withdraw(self, amount):
        if amount < 0:
            raise ValueError("Withdrawal amount must be non-negative.")
        if amount > self.balance:
            raise ValueError("Insufficient funds for this withdrawal.")
        self.balance -= amount
        return self.balance

    def __str__(self):
        return f"{self.account_number} | {self.holder} | {self.balance:,.2f} {self.currency}"

In [ ]:
account = BankAccount("EE3800123456", "Marwa Tamam", balance=4520.50)
print(account)
account.deposit(500)
account.withdraw(120)
print("Balance after transactions:", account.balance)

try:
    account.withdraw(5_000)
except ValueError as error:
    print("Blocked:", error)

assert account.balance == 4900.50

### Guided activity 1 — build a `WorkItem` class

Create a class with `case_id`, `priority`, `status`, and `hours_open`. Add methods that:

1. Mark the item as `closed`.
2. Return `True` when the item exceeds a supplied SLA.
3. Display a readable summary.

Use one object to test each method.

In [ ]:
# TODO: define your class and test it here


## 2. Attributes, class attributes, and validation

An instance attribute belongs to one object. A class attribute is shared by the class and provides a common default.

The queue below uses a class attribute for its default capacity and validates every addition.

In [ ]:
class WorkQueue:
    default_limit = 3

    def __init__(self, name, limit=None):
        self.name = name
        self.limit = limit if limit is not None else WorkQueue.default_limit
        self.items = []

    def add(self, item):
        if len(self.items) >= self.limit:
            raise ValueError("Queue is full.")
        self.items.append(item)
        return len(self.items)

    def open_items(self):
        return [item for item in self.items if item.status in WorkItem.open_statuses]


## 3. Decorators: adding reusable behavior

A decorator receives a function and returns a new function with extra behavior. The `@decorator` syntax applies it when the function is defined.

A reusable decorator usually accepts the original function, defines a wrapper, preserves metadata with `@wraps`, calls the original function, and returns its result.

In [ ]:
# Let's see this simple function:

def calculate_total(price, tax):
    return price + tax

print(calculate_total(20, 2))
print(calculate_total.__name__)

In [ ]:
import time

def calculate_total_2(price, tax):
    time.sleep(5)  # Simulate some processing time
    return price + tax

print(calculate_total_2(20, 2))

In [ ]:
def sleep(func):
    def wrapper(*args, **kwargs):
        time.sleep(5)  # Simulate some processing time
        return func(*args, **kwargs)
    return wrapper


In [ ]:
@sleep
def calculate_invoice(price, tax):
    price *= tax
    return (round(price, 2))

calculate_invoice(12_000, 0.14)

In [ ]:
from functools import wraps

def announce(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f"Starting {func.__name__}")
        result = func(*args, **kwargs)
        print(f"Finished {func.__name__}")
        return result
    return wrapper

@announce
def make_label(case_id, status="open"):
    return f"{case_id}: {status}"

print(make_label("A101", status="closed"))
print("Function name:", make_label.__name__)

### Why `*args`, `**kwargs`, and `@wraps` matter

`*args` accepts positional arguments and `**kwargs` accepts keyword arguments. Without `**kwargs`, a decorated function may fail when called with named arguments. `@wraps` keeps the original function's name and documentation.

In [ ]:
def log_call(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f"Running {func.__name__} with args={args}, kwargs={kwargs}")
        return func(*args, **kwargs)
    return wrapper

@log_call
def calculate_total(price, tax):
    return price + tax

assert calculate_total(price=20, tax=2) == 22
print(calculate_total.__name__)

In [ ]:
# What about that: :D

def log_call(func):
    def wrapper(*args, **kwargs):
        print(f"Running {func.__name__}")
        return func(*args, **kwargs)
    return wrapper

@log_call
def calculate_total(price, tax):
    return price + tax

print(calculate_total(20, 2))
print(calculate_total.__name__)

### Guided activity 2 — write a decorator

Write a `log_call` decorator that prints the function name and arguments, returns the original result, and preserves the function name. Apply it to a function such as `calculate_sla(hours_open, sla_hours)`.

In [ ]:
# TODO: write and apply your decorator


## 4. Practical decorators

Decorators are useful for cross-cutting behavior such as validation, timing, logging, and caching. The decorated function still performs its main job; the decorator adds reusable behavior around it.

In [ ]:
import time

def timer(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"{func.__name__} took {elapsed:.6f} seconds")
        return result
    return wrapper

def require_positive(func):
    @wraps(func)
    def wrapper(amount, *args, **kwargs):
        if amount <= 0:
            raise ValueError("Amount must be positive.")
        return func(amount, *args, **kwargs)
    return wrapper

@timer
@require_positive
def sum_squares(amount):
    return sum(number ** 2 for number in range(amount))

print(sum_squares(1000))
try:
    sum_squares(0)
except ValueError as error:
    print("Blocked:", error)

### Guided activity 4 — decorators on methods

Apply a decorator to a class method. Remember that a method receives `self` first. Create a decorator that prevents a work item from being updated with a negative number of hours. Test a valid and invalid update.

In [ ]:
# TODO: create a method decorator and apply it to a class method


## 5. Data classes

A **data class** is a class designed mainly to store related data. The `@dataclass` decorator can generate `__init__()`, `__repr__()`, and `__eq__()` from annotated fields.

> **Key idea:** `@dataclass` is a class-building decorator. It is different from function decorators, which wrap a function to add behavior.

In [ ]:
from dataclasses import dataclass


@dataclass
class Ticket:
    identifier: int
    title: str
    status: str


# The dataclass decorator provides an initializer:
ticket = Ticket(1, "Fix login issue", "open")

# It provides a readable representation:
print(ticket)
# Ticket(identifier=1, title='Fix login issue', status='open')

# It provides equality based on the field values:
same_ticket = Ticket(1, "Fix login issue", "open")
print(ticket == same_ticket)
# True

### Guided activity 5 — define a data class

Define a `Ticket` data class with an identifier, a title, and a status. Add an `is_closed()` method. Create two objects, compare them, and add `assert` checks.

In [ ]:
# TODO: define and test a Ticket data class


<details>
<summary>💡 Show solution</summary>

```python
from dataclasses import dataclass

@dataclass
class Ticket:
    identifier: int
    title: str
    status: str

    def is_closed(self) -> bool:
        return self.status == "closed"


first_ticket = Ticket(1, "Fix login issue", "open")
second_ticket = Ticket(2, "Update help page", "closed")

print(first_ticket)
print(second_ticket)
print("Are the tickets equal?", first_ticket == second_ticket)
print("Is the first ticket closed?", first_ticket.is_closed())
print("Is the second ticket closed?", second_ticket.is_closed())

assert first_ticket != second_ticket
assert first_ticket.is_closed() is False
assert second_ticket.is_closed() is True

## 6. Practice tasks

Complete the tasks independently. Add a small test after each non-trivial class or decorator.

### Applied task A — `S10-E01`
Create a `WorkItem` class with four attributes, a `close()` method, an `is_overdue()` method, and a readable `__str__()` method.

In [ ]:
# Your code here


### Applied task B — `S10-E02`
Create a `WorkQueue` class that stores work items, rejects items after a maximum capacity, and returns only open items.

In [ ]:
# Your code here


### Applied task C — `S10-E03`
Write an `announce` or `log_call` decorator. Apply it to two functions with different arguments and confirm that both return their normal results.

In [ ]:
# Your code here


### Applied task D — `S10-E04`
Write a decorator that rejects non-positive numeric input. Apply it to a function that calculates an SLA percentage or a total amount.

In [ ]:
# Your code here


### Challenge — `S10-E05`
Use two decorators on one function: one for timing and one for logging. Explain which decorator runs first and why `@wraps` matters.

In [ ]:
# Your code here


## 7. Guided mini-project — decorated work-item queue

Build a small work-item processing system. Your solution must:

1. Define a `WorkItem` class.
2. Define a `WorkQueue` class that adds and closes items.
3. Validate at least one state-changing operation.
4. Add a logging decorator to one queue method.
5. Add a timing decorator to one function or method.
6. Produce a summary of open, closed, and overdue items.
7. Include at least one `assert` self-check.

Keep the classes small. The goal is to demonstrate object state and reusable decorators, not to build a production queue.

In [ ]:
# TODO: complete the mini-project


## 8. Exit check — answer from memory

1. What is a class?
2. What is an object?
3. What does `self` refer to?
4. What is the purpose of `__init__`?
5. What is the difference between an instance attribute and a class attribute?
6. What does `@dataclass` generate for a class with annotated fields?
7. Why use `field(default_factory=list)` instead of `tags=[]`?
8. What does a decorator receive and return?
9. Why is `functools.wraps` useful?
10. Why do reusable decorators use `*args` and `**kwargs`?
11. What happens when multiple decorators are applied to one function?

## Common mistakes and next steps

- Remember `self` in instance methods.
- Use instance attributes for per-object state.
- Keep validation beside the state-changing method.
- Use `@dataclass` for data-focused classes with annotated fields.
- Return the original function's result from a decorator.
- Use `@wraps` in reusable decorators.
- Use `*args` and `**kwargs` when a decorator should support different signatures.
- Test invalid inputs as well as successful inputs.

**Next step:** use classes to model larger systems and decorators to add consistent logging, validation, timing, or caching behavior.